# 04 - Key Findings (reproducible)

This notebook computes the four headline findings in `REPORT.md` directly from the engine and the shipped data, so every number in the report is backed by runnable code.

1. **Discrimination** on a real control group (12 healthy vs 9 distressed Indian firms)
2. **Lead time** over Altman on the Track Record collapses
3. **Hard-event floor** ablation (does the safeguard actually bind?)
4. **Weight robustness** of the 60/40 financial/signal blend

Run top to bottom. It needs only `requirements.txt` (no lightgbm/shap/optuna).

In [1]:
import sys, csv, pathlib, datetime as dt
import numpy as np
REPO = pathlib.Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
import foresight as f
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
print('engine loaded; roster =', [fn.company for fn,_,_ in f.ROSTER])

engine loaded; roster = ['SpiceJet', 'Ola Electric', 'Vodafone Idea', 'Vedanta', 'Paytm', 'TCS']


## Finding 1 - Discrimination on a real control group

The `data/indian/companies.csv` set is a genuine control group: 12 financially healthy blue-chips and 9 firms that went to NCLT or default. We re-fit the shipped logistic (Altman's four ratios) with leave-one-out CV and ask a credit-desk question: at a threshold that raises **zero false alarms** on the healthy firms, how many of the failures do we still catch?

In [2]:
FEATS = ['Attr3', 'Attr6', 'Attr7', 'Attr8']
rows = list(csv.DictReader(open(REPO/'data'/'indian'/'companies.csv', encoding='utf-8')))
def feats(r):
    num = lambda k: float(r[k])
    fin = f.ScreenerFinancials(company=r['company'], year=int(r['year']),
        sales=num('sales'), expenses=num('expenses'), operating_profit=num('operating_profit'),
        other_income=num('other_income'), interest=num('interest'), depreciation=num('depreciation'),
        profit_before_tax=num('profit_before_tax'), net_profit=num('net_profit'),
        equity_capital=num('equity_capital'), reserves=num('reserves'), borrowings=num('borrowings'),
        other_liabilities=num('other_liabilities'), total_assets=num('total_assets'),
        fixed_assets=num('fixed_assets'), working_capital_days=num('working_capital_days'))
    fe = f.compute_features(fin)
    return [fe.get(c, np.nan) for c in FEATS]
X = np.array([feats(r) for r in rows], 'float64')
y = np.array([int(r['label']) for r in rows], 'int8')
pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                     LogisticRegression(class_weight='balanced', max_iter=3000))
oof = cross_val_predict(pipe, X, y, cv=LeaveOneOut(), method='predict_proba')[:, 1]
print(f'n={len(y)}  healthy={(y==0).sum()}  distressed={(y==1).sum()}')
print(f'Leave-one-out ROC-AUC = {roc_auc_score(y, oof):.4f}   PR-AUC = {average_precision_score(y, oof):.4f}')
print(f'median P(distress):  healthy = {np.median(oof[y==0]):.2f}   distressed = {np.median(oof[y==1]):.2f}')
thr = oof[y==0].max()  # highest score among healthy firms -> a cut above it flags 0 healthy
caught = int((oof[y==1] > thr).sum())
print(f'at a cut that flags 0/{(y==0).sum()} healthy firms: catches {caught}/{(y==1).sum()} distressed')

n=21  healthy=12  distressed=9
Leave-one-out ROC-AUC = 0.9722   PR-AUC = 0.9722
median P(distress):  healthy = 0.13   distressed = 0.83
at a cut that flags 0/12 healthy firms: catches 8/9 distressed


## Finding 2 - Lead time over Altman

For each collapse in the Track Record we take the first date each score crosses into **high risk (>= 70)** *before* the insolvency event, and measure the warning in months. The trajectories mirror `app/main.py::TRACK_RECORD` (Altman is the financial-only leg; the comprehensive score adds the signals). These are reconstructed on each firm's real history, so this is illustrative - Finding 1 is the out-of-sample evidence.

In [3]:
# (name, event_date, altman [(fiscal_year, risk)], comprehensive [(iso_date, score)])
TR = {
 'Unitech': ('2020-01-20',
    [(2015,20),(2016,20),(2017,21),(2018,29),(2019,34),(2021,63),(2023,85)],
    [('2015-03-31',44),('2016-03-31',48),('2017-04-26',66),('2017-12-01',72),
     ('2018-03-31',74),('2019-03-31',77),('2020-01-20',83),('2021-06-30',86),('2023-03-31',90)]),
 'Future Retail': ('2022-07-20',
    [(2019,15),(2020,45),(2021,85)],
    [('2019-03-31',34),('2019-08-22',44),('2020-03-31',60),('2020-10-25',74),
     ('2021-03-31',86),('2021-12-24',90),('2022-07-20',95)]),
 'Reliance Comm': ('2019-02-01',
    [(2015,34),(2016,60),(2017,75),(2018,95),(2019,96)],
    [('2015-03-31',44),('2015-06-30',50),('2016-03-31',66),('2016-09-05',72),
     ('2017-03-31',80),('2017-05-30',84),('2017-09-15',88),('2018-03-31',95),('2019-02-01',97)]),
 'Jaiprakash': ('2024-06-03',
    [(2016,39),(2017,82),(2018,50),(2019,67),(2020,37),(2021,42),(2022,46),(2023,44),(2024,49)],
    [('2016-03-31',52),('2017-03-31',86),('2017-06-01',85),('2018-03-31',74),('2018-09-01',78),
     ('2020-03-31',74),('2021-03-01',80),('2022-09-01',84),('2024-06-03',90)]),
}
THR = 70.0
mo = lambda a, b: (b.year-a.year)*12 + (b.month-a.month) + (b.day-a.day)/30.0
comp_lead, extra = [], []
for name,(ev,alt,fore) in TR.items():
    event = dt.date.fromisoformat(ev)
    fd = next((dt.date.fromisoformat(d) for d,s in fore if s>=THR and dt.date.fromisoformat(d)<event), None)
    ad = next((dt.date(yr,3,31) for yr,r in alt if r>=THR and dt.date(yr,3,31)<event), None)
    mf = mo(fd, event) if fd else 0.0
    ma = mo(ad, event) if ad else None
    comp_lead.append(mf); extra.append(mf - (ma or 0))
    print(f'{name:15} comprehensive {mf:5.0f} mo before | Altman {("%5.0f mo"%ma) if ma is not None else "   never":>8} | extra {mf-(ma or 0):5.0f} mo')
print(f'\nmedian comprehensive warning = {np.median(comp_lead):.0f} months before insolvency')
print(f'median EXTRA lead over Altman = {np.median(extra):.0f} months')

Unitech         comprehensive    26 mo before | Altman    never | extra    26 mo
Future Retail   comprehensive    21 mo before | Altman    16 mo | extra     5 mo
Reliance Comm   comprehensive    29 mo before | Altman    22 mo | extra     7 mo
Jaiprakash      comprehensive    86 mo before | Altman    86 mo | extra     0 mo

median comprehensive warning = 27 months before insolvency
median EXTRA lead over Altman = 6 months


## Finding 3 - Does the hard-event floor actually bind?

The floor lets a *verified fact* (auditor exit, board suspension) set a risk floor the softer, tone-based signals cannot average away. We compare each roster company's digital pulse **with** the floor (as shipped) against the plain weighted average **without** it.

In [4]:
AS_OF = f.AS_OF
for name in f._DATA:
    p = f.pulse_as_of(name, AS_OF)
    present = [r for r in p.readings if r is not None]
    tw = sum(f._WEIGHTS.get(r.kind, 0.0) for r in present)
    unfloored = sum(r.risk_score*f._WEIGHTS.get(r.kind, 0.0) for r in present)/tw if tw else 0.0
    hard = [round(r.risk_score) for r in present if r.hard_event]
    bind = '  <-- FLOOR BINDS' if p.composite_score > unfloored + 0.05 else ''
    print(f'{name:15} without floor = {unfloored:5.1f}   with floor = {p.composite_score:5.1f}   hard-events={hard}{bind}')

SpiceJet        without floor =  76.2   with floor =  78.0   hard-events=[78]  <-- FLOOR BINDS
Ola Electric    without floor =  58.5   with floor =  58.5   hard-events=[]
Vodafone Idea   without floor =  53.0   with floor =  53.0   hard-events=[]
Vedanta         without floor =  13.4   with floor =  13.4   hard-events=[]
Paytm           without floor =  21.1   with floor =  21.1   hard-events=[]
TCS             without floor =  22.1   with floor =  22.1   hard-events=[]


## Finding 4 - Is the 60/40 blend load-bearing?

We re-rank the roster by the combined score under three financial/signal splits. If the ordering is stable, the exact weights are not what drives the conclusion - the signals are.

In [5]:
def combined(name, fw, dw):
    fin = next(fn for fn,_,_ in f.ROSTER if fn.company == name)
    return f.fuse(f.score_company(fin), f.pulse_as_of(name, AS_OF), fw, dw).combined_score
names = list(f._DATA)
splits = {'60/40 (shipped)': (0.6,0.4), '50/50': (0.5,0.5), '70/30': (0.7,0.3)}
order = {s: sorted(names, key=lambda n: -combined(n, fw, dw)) for s,(fw,dw) in splits.items()}
for s in splits:
    print(f'{s:16} worst -> best: ' + '  >  '.join(order[s]))
base = order['60/40 (shipped)']
for s in ['50/50','70/30']:
    print(f'ranking under {s} vs shipped: ' + ('IDENTICAL' if order[s]==base else 'CHANGED'))

60/40 (shipped)  worst -> best: SpiceJet  >  Vodafone Idea  >  Ola Electric  >  Vedanta  >  TCS  >  Paytm
50/50            worst -> best: SpiceJet  >  Vodafone Idea  >  Ola Electric  >  Vedanta  >  TCS  >  Paytm
70/30            worst -> best: SpiceJet  >  Vodafone Idea  >  Ola Electric  >  Vedanta  >  TCS  >  Paytm
ranking under 50/50 vs shipped: IDENTICAL
ranking under 70/30 vs shipped: IDENTICAL


## Summary

| Finding | Result |
|---|---|
| Discrimination (control group) | median P 0.83 vs 0.13; catches 8/9 at 0 false alarms; LOO ROC-AUC 0.97 |
| Lead time vs Altman | median 27 mo warning; +6 mo over Altman; Unitech +26 mo where Altman never warned |
| Hard-event floor | binds on SpiceJet (76 -> 78 via the auditor exit); inert on the healthy names |
| Weight robustness | ranking identical at 50/50, 60/40, 70/30 |

The financial engine is Altman; the added value is breadth, explainability, and validation - and these four results quantify each part.